# Training Hub Unified Callbacks — Example Notebook

This notebook validates unified `TrainingHubCallback` support across Training Hub backends on Red Hat OpenShift AI. It submits six TrainJobs — a baseline run and a callback-enabled run for each backend — and verifies SDK injection and hook output in pod logs.

| # | Backend | Algorithm | `callbacks=` | Expected log behavior |
|---|---------|-----------|--------------|------------------------|
| 1 | Unsloth | `lora_sft` | omitted | Training completes; no SDK injection lines; no callback hook markers |
| 2 | Unsloth | `lora_sft` | `UnslothSmokeLogger` | `[UNSLOTH\|CALLBACK\|...]` markers |
| 3 | InstructLab | `sft` | omitted | Same baseline behavior as row 1 |
| 4 | InstructLab | `sft` | `InstructLabSmokeLogger` | `[ILAB\|CALLBACK\|...]` markers |
| 5 | MiniTrainer | `osft` | omitted | Same baseline behavior as row 1 |
| 6 | MiniTrainer | `osft` | `MiniTrainerSmokeLogger` | `[MINI\|CALLBACK\|...]` markers |

**Execution:** Kernel → Restart & Run All (approximately 1–2 hours for all six jobs).

**Prerequisites:** Kubeflow Trainer on cluster, GPU runtime, `training_hub` with unified callbacks.

## 1. Install SDK branch (run once per environment)

Uses a feature branch until callbacks land in a stable SDK release.

In [ ]:
# Install the SDK with callbacks support.
# TODO: replace with stable release once callbacks land in a released SDK version.
%pip install -q "git+https://github.com/opendatahub-io/kubeflow-sdk.git@feat/rhoaieng-79848-sdk-callbacks"

import inspect

from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TrainingHubAlgorithms, TrainingHubTrainer

assert "callbacks" in inspect.signature(TrainingHubTrainer).parameters, (
    "TrainingHubTrainer.callbacks missing — reinstall the SDK branch with callbacks support"
)

client = TrainerClient()
print("SDK OK — TrainingHubTrainer.callbacks is available")
print("Runtimes:")
for runtime in client.list_runtimes():
    print(f"  - {runtime.name}")

## 2. Configuration

Git refs below use branch tips (`@main`, feature branches) on purpose — this example tracks unreleased callback support. Replace with release tags once the SDK and Training Hub ship stable versions.

In [ ]:
import os
from pathlib import Path

RUNTIME_NAME = os.environ.get("TRAINING_RUNTIME", "training-hub")
TRAINING_HUB_REF = os.environ.get(
    "TRAINING_HUB_REF",
    "git+https://github.com/Red-Hat-AI-Innovation-Team/training_hub.git@main",
)
MINI_TRAINER_REF = os.environ.get(
    "MINI_TRAINER_REF",
    "git+https://github.com/Red-Hat-AI-Innovation-Team/mini_trainer.git@main",
)

MODEL_PATH = os.environ.get("SMOKE_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
WORKSPACE = Path(os.environ.get("WORKSPACE_DIR", "/opt/app-root/src"))
HF_TOKEN = os.environ.get("HF_TOKEN", "")

CPU = os.environ.get("SMOKE_CPU", "4")
MEMORY = os.environ.get("SMOKE_MEMORY", "16Gi")
GPU = os.environ.get("SMOKE_GPU", "1")

COMMON_ENV = {
    "HF_HOME": str(WORKSPACE / ".cache" / "huggingface"),
    "TOKENIZERS_PARALLELISM": "false",
    **({"HF_TOKEN": HF_TOKEN} if HF_TOKEN else {}),
}
COMMON_RESOURCES = {"cpu": CPU, "memory": MEMORY, "nvidia.com/gpu": GPU}

print(f"runtime={RUNTIME_NAME}")
print(f"model={MODEL_PATH}")

## 3. Write helper modules

Callback + training funcs must be regular `.py` files (SDK uses `inspect.getsource`).

In [ ]:
%%writefile hub_callback_smoke.py
"""Training Hub callbacks with backend-specific log tags for verification."""

from training_hub import TrainingHubCallback, TrainingHubContext


class _TaggedSmokeLogger(TrainingHubCallback):
    TAG: str = "GENERIC"

    def on_train_begin(self, context: TrainingHubContext) -> None:
        print(
            f"[{self.TAG}|CALLBACK|BEGIN] backend={self.TAG} "
            f"output_dir={context.output_dir} main={context.is_main_process}",
            flush=True,
        )

    def on_log(self, context: TrainingHubContext) -> None:
        print(
            f"[{self.TAG}|CALLBACK|LOG] step={context.step} epoch={context.epoch} "
            f"loss={context.loss} lr={context.learning_rate}",
            flush=True,
        )

    def on_train_end(self, context: TrainingHubContext) -> None:
        print(
            f"[{self.TAG}|CALLBACK|END] step={context.step} backend={self.TAG}",
            flush=True,
        )


class UnslothSmokeLogger(_TaggedSmokeLogger):
    TAG = "UNSLOTH"


class InstructLabSmokeLogger(_TaggedSmokeLogger):
    TAG = "ILAB"


class MiniTrainerSmokeLogger(_TaggedSmokeLogger):
    TAG = "MINI"


UNSLOTH_CALLBACK_MARKERS = ["[UNSLOTH|CALLBACK|BEGIN]", "[UNSLOTH|CALLBACK|LOG]", "[UNSLOTH|CALLBACK|END]"]
ILAB_CALLBACK_MARKERS = ["[ILAB|CALLBACK|BEGIN]", "[ILAB|CALLBACK|LOG]", "[ILAB|CALLBACK|END]"]
MINI_CALLBACK_MARKERS = ["[MINI|CALLBACK|BEGIN]", "[MINI|CALLBACK|LOG]", "[MINI|CALLBACK|END]"]

In [ ]:
%%writefile smoke_train_func.py
"""Unsloth backend — calls training_hub.lora_sft()."""


def callback_smoke_train(**kwargs) -> None:
    import json
    from pathlib import Path

    from training_hub import lora_sft

    data_path = Path("/tmp/callback_smoke_data.jsonl")
    rows = [
        {"messages": [{"role": "user", "content": "What is 2+2?"}, {"role": "assistant", "content": "4"}]},
        {"messages": [{"role": "user", "content": "Capital of France?"}, {"role": "assistant", "content": "Paris"}]},
        {"messages": [{"role": "user", "content": "Say hello"}, {"role": "assistant", "content": "Hello!"}]},
        {"messages": [{"role": "user", "content": "Color of the sky?"}, {"role": "assistant", "content": "Blue"}]},
    ]
    with data_path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")

    args = dict(kwargs)
    args["data_path"] = str(data_path)
    args.setdefault("ckpt_output_dir", "/tmp/callback_smoke_out")
    print(f"[PY] Launching lora_sft (Unsloth) data={data_path}", flush=True)
    lora_sft(**args)
    print("[PY] lora_sft complete", flush=True)

In [ ]:
%%writefile smoke_train_func_sft.py
"""InstructLab backend — calls training_hub.sft()."""


def callback_smoke_train_sft(**kwargs) -> None:
    import json
    from pathlib import Path

    from training_hub import sft

    data_path = Path("/tmp/callback_smoke_data_sft.jsonl")
    rows = [
        {"messages": [{"role": "user", "content": "What is 2+2?"}, {"role": "assistant", "content": "4"}]},
        {"messages": [{"role": "user", "content": "Capital of France?"}, {"role": "assistant", "content": "Paris"}]},
        {"messages": [{"role": "user", "content": "Say hello"}, {"role": "assistant", "content": "Hello!"}]},
        {"messages": [{"role": "user", "content": "Color of the sky?"}, {"role": "assistant", "content": "Blue"}]},
    ]
    with data_path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")

    args = dict(kwargs)
    args["data_path"] = str(data_path)
    args.setdefault("ckpt_output_dir", "/tmp/callback_smoke_out_sft")
    args.setdefault("max_tokens_per_gpu", 4096)
    args.setdefault("mlflow_tracking_uri", "")
    print(f"[PY] Launching sft (InstructLab) data={data_path}", flush=True)
    sft(**args)
    print("[PY] sft complete", flush=True)

In [ ]:
%%writefile smoke_train_func_osft.py
"""MiniTrainer backend — calls training_hub.osft()."""


def callback_smoke_train_osft(**kwargs) -> None:
    import json
    from pathlib import Path

    from training_hub import osft

    data_path = Path("/tmp/callback_smoke_data_osft.jsonl")
    rows = [
        {"messages": [{"role": "user", "content": "What is 2+2?"}, {"role": "assistant", "content": "4"}]},
        {"messages": [{"role": "user", "content": "Capital of France?"}, {"role": "assistant", "content": "Paris"}]},
        {"messages": [{"role": "user", "content": "Say hello"}, {"role": "assistant", "content": "Hello!"}]},
        {"messages": [{"role": "user", "content": "Color of the sky?"}, {"role": "assistant", "content": "Blue"}]},
    ]
    with data_path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")

    args = dict(kwargs)
    args["data_path"] = str(data_path)
    args.setdefault("ckpt_output_dir", "/tmp/callback_smoke_out_osft")
    args.setdefault("unfreeze_rank_ratio", 0.1)
    args.setdefault("max_tokens_per_gpu", 4096)
    args.setdefault("mlflow_tracking_uri", "")
    print(f"[PY] Launching osft (MiniTrainer) data={data_path}", flush=True)
    osft(**args)
    print("[PY] osft complete", flush=True)

## 4. Shared helpers

Submit a job, wait for completion, fetch logs, and verify callback markers.

In [ ]:
from kubeflow.trainer.constants import constants
from kubeflow.trainer.rhai import TrainingHubTrainer

SDK_MARKERS = [
    "Prepared 1 Training Hub callback",
    "Training Hub callback injection configured",
]

VERIFICATION_RESULTS: dict[str, bool] = {}


def wait_and_get_logs(job_name: str, timeout: int = 3600) -> tuple[bool, str]:
    """Wait for TrainJob completion and return (completed, log_text)."""
    completed = False
    try:
        job = client.wait_for_job_status(
            name=job_name,
            status={constants.TRAINJOB_COMPLETE},
            timeout=timeout,
            polling_interval=10,
        )
        print(f"TrainJob {job_name} status: {job.status}")
        completed = True
    except Exception as exc:
        print(f"TrainJob {job_name} did not complete: {exc}")
    logs = "".join(client.get_job_logs(name=job_name, follow=False, step="node-0"))
    return completed, logs


def verify_logs(
    logs: str,
    *,
    label: str,
    expect_callbacks: bool,
    result_key: str,
    callback_markers: list[str],
    job_completed: bool,
) -> bool:
    """Verify pod logs match baseline or callback-enabled expectations."""
    has_sdk = all(marker in logs for marker in SDK_MARKERS)
    has_hooks = all(marker in logs for marker in callback_markers)

    print(f"\n{'=' * 60}")
    print(label)
    print(f"{'=' * 60}")
    print(f"Job completed: {job_completed}")
    print(f"Expect callbacks: {expect_callbacks}")

    print("\nSDK injection markers:")
    for marker in SDK_MARKERS:
        found = marker in logs
        print(f"  {'PASS' if found else 'absent'}  {marker}")

    print("\nBackend callback markers:")
    for marker in callback_markers:
        found = marker in logs
        print(f"  {'PASS' if found else 'absent'}  {marker}")

    if not job_completed:
        passed = False
        print("\nRESULT: ❌ FAILED — job did not complete successfully")
    elif expect_callbacks:
        passed = has_sdk and has_hooks
        print(
            f"\nRESULT: {'✅ WITH CALLBACKS OK' if passed else '❌ WITH CALLBACKS FAILED'}"
        )
    else:
        passed = not any(m in logs for m in SDK_MARKERS) and not any(
            m in logs for m in callback_markers
        )
        print(
            f"\nRESULT: {'✅ BASELINE OK (no callbacks)' if passed else '❌ BASELINE UNEXPECTED MARKERS'}"
        )

    VERIFICATION_RESULTS[result_key] = passed
    return passed


def submit_and_verify(
    trainer: TrainingHubTrainer,
    *,
    label: str,
    expect_callbacks: bool,
    result_key: str,
    callback_markers: list[str],
) -> str:
    """Submit TrainJob, wait, verify logs. Returns job name."""
    job_name = client.train(runtime=RUNTIME_NAME, trainer=trainer)
    print(f"Submitted: {job_name}")
    completed, logs = wait_and_get_logs(job_name)
    print("\n--- pod logs (tail) ---")
    tail = logs[-4000:] if len(logs) > 4000 else logs
    print(tail)
    verify_logs(
        logs,
        label=label,
        expect_callbacks=expect_callbacks,
        result_key=result_key,
        callback_markers=callback_markers,
        job_completed=completed,
    )
    return job_name

---
# Part A — Unsloth (`lora_sft`)

**Baseline:** no `callbacks=` on the trainer; training runs without callback hook output.

**With callbacks:** `callbacks=[UnslothSmokeLogger]`; pod logs include `[UNSLOTH|CALLBACK|...]` markers.

### A1 · Baseline — Unsloth without callbacks

In [ ]:
from hub_callback_smoke import UNSLOTH_CALLBACK_MARKERS
from smoke_train_func import callback_smoke_train

trainer_unsloth_before = TrainingHubTrainer(
    func=callback_smoke_train,
    algorithm=TrainingHubAlgorithms.LORA_SFT,
    func_args={
        "model_path": MODEL_PATH,
        "ckpt_output_dir": "/tmp/callback_smoke_out_before",
        "num_epochs": 1,
        "max_seq_len": 128,
        "micro_batch_size": 1,
        "logging_steps": 1,
        "save_steps": 9999,
        "save_total_limit": 1,
        "warmup_steps": 0,
        "learning_rate": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "sample_packing": True,
    },
    packages_to_install=[TRAINING_HUB_REF],
    resources_per_node=COMMON_RESOURCES,
    enable_progression_tracking=False,
    env=COMMON_ENV,
)

unsloth_before_job = submit_and_verify(
    trainer_unsloth_before,
    label="A1 · Unsloth (lora_sft) — Baseline (no callbacks=)",
    expect_callbacks=False,
    result_key="unsloth_before",
    callback_markers=UNSLOTH_CALLBACK_MARKERS,
)

### A2 · With callbacks — Unsloth with unified callback

In [ ]:
from hub_callback_smoke import UNSLOTH_CALLBACK_MARKERS, UnslothSmokeLogger

trainer_unsloth_after = TrainingHubTrainer(
    func=callback_smoke_train,
    algorithm=TrainingHubAlgorithms.LORA_SFT,
    func_args={
        "model_path": MODEL_PATH,
        "ckpt_output_dir": "/tmp/callback_smoke_out_after",
        "num_epochs": 1,
        "max_seq_len": 128,
        "micro_batch_size": 1,
        "logging_steps": 1,
        "save_steps": 9999,
        "save_total_limit": 1,
        "warmup_steps": 0,
        "learning_rate": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "sample_packing": True,
    },
    callbacks=[UnslothSmokeLogger],
    packages_to_install=[TRAINING_HUB_REF],
    resources_per_node=COMMON_RESOURCES,
    enable_progression_tracking=False,
    env=COMMON_ENV,
)

unsloth_after_job = submit_and_verify(
    trainer_unsloth_after,
    label="A2 · Unsloth (lora_sft) — With callbacks (UnslothSmokeLogger)",
    expect_callbacks=True,
    result_key="unsloth_after",
    callback_markers=UNSLOTH_CALLBACK_MARKERS,
)

---
# Part B — InstructLab Training (`sft`)

### B1 · Baseline — InstructLab without callbacks

In [ ]:
from hub_callback_smoke import ILAB_CALLBACK_MARKERS
from smoke_train_func_sft import callback_smoke_train_sft

trainer_sft_before = TrainingHubTrainer(
    func=callback_smoke_train_sft,
    algorithm=TrainingHubAlgorithms.SFT,
    func_args={
        "model_path": MODEL_PATH,
        "ckpt_output_dir": "/tmp/callback_smoke_out_sft_before",
        "num_epochs": 1,
        "max_seq_len": 128,
        "effective_batch_size": 4,
        "learning_rate": 2e-5,
        "max_tokens_per_gpu": 4096,
        "mlflow_tracking_uri": "",
    },
    packages_to_install=[TRAINING_HUB_REF],
    resources_per_node=COMMON_RESOURCES,
    enable_progression_tracking=False,
    env=COMMON_ENV,
)

sft_before_job = submit_and_verify(
    trainer_sft_before,
    label="B1 · InstructLab (sft) — Baseline (no callbacks=)",
    expect_callbacks=False,
    result_key="sft_before",
    callback_markers=ILAB_CALLBACK_MARKERS,
)

### B2 · With callbacks — InstructLab with unified callback

In [ ]:
from hub_callback_smoke import ILAB_CALLBACK_MARKERS, InstructLabSmokeLogger

trainer_sft_after = TrainingHubTrainer(
    func=callback_smoke_train_sft,
    algorithm=TrainingHubAlgorithms.SFT,
    func_args={
        "model_path": MODEL_PATH,
        "ckpt_output_dir": "/tmp/callback_smoke_out_sft_after",
        "num_epochs": 1,
        "max_seq_len": 128,
        "effective_batch_size": 4,
        "learning_rate": 2e-5,
        "max_tokens_per_gpu": 4096,
        "mlflow_tracking_uri": "",
    },
    callbacks=[InstructLabSmokeLogger],
    packages_to_install=[TRAINING_HUB_REF],
    resources_per_node=COMMON_RESOURCES,
    enable_progression_tracking=False,
    env=COMMON_ENV,
)

sft_after_job = submit_and_verify(
    trainer_sft_after,
    label="B2 · InstructLab (sft) — With callbacks (InstructLabSmokeLogger)",
    expect_callbacks=True,
    result_key="sft_after",
    callback_markers=ILAB_CALLBACK_MARKERS,
)

---
# Part C — MiniTrainer (`osft`)

OSFT with callbacks needs `MINI_TRAINER_REF` installed from source (runtime image does not yet include the fix).

### C1 · Baseline — MiniTrainer without callbacks

In [ ]:
from hub_callback_smoke import MINI_CALLBACK_MARKERS
from smoke_train_func_osft import callback_smoke_train_osft

trainer_osft_before = TrainingHubTrainer(
    func=callback_smoke_train_osft,
    algorithm=TrainingHubAlgorithms.OSFT,
    func_args={
        "model_path": MODEL_PATH,
        "ckpt_output_dir": "/tmp/callback_smoke_out_osft_before",
        "num_epochs": 1,
        "max_seq_len": 128,
        "effective_batch_size": 4,
        "learning_rate": 2e-5,
        "unfreeze_rank_ratio": 0.1,
        "max_tokens_per_gpu": 4096,
        "mlflow_tracking_uri": "",
    },
    packages_to_install=[TRAINING_HUB_REF, MINI_TRAINER_REF],
    resources_per_node=COMMON_RESOURCES,
    enable_progression_tracking=False,
    env=COMMON_ENV,
)

osft_before_job = submit_and_verify(
    trainer_osft_before,
    label="C1 · MiniTrainer (osft) — Baseline (no callbacks=)",
    expect_callbacks=False,
    result_key="osft_before",
    callback_markers=MINI_CALLBACK_MARKERS,
)

### C2 · With callbacks — MiniTrainer with unified callback

In [ ]:
from hub_callback_smoke import MINI_CALLBACK_MARKERS, MiniTrainerSmokeLogger

trainer_osft_after = TrainingHubTrainer(
    func=callback_smoke_train_osft,
    algorithm=TrainingHubAlgorithms.OSFT,
    func_args={
        "model_path": MODEL_PATH,
        "ckpt_output_dir": "/tmp/callback_smoke_out_osft_after",
        "num_epochs": 1,
        "max_seq_len": 128,
        "effective_batch_size": 4,
        "learning_rate": 2e-5,
        "unfreeze_rank_ratio": 0.1,
        "max_tokens_per_gpu": 4096,
        "mlflow_tracking_uri": "",
    },
    callbacks=[MiniTrainerSmokeLogger],
    packages_to_install=[TRAINING_HUB_REF, MINI_TRAINER_REF],
    resources_per_node=COMMON_RESOURCES,
    enable_progression_tracking=False,
    env=COMMON_ENV,
)

osft_after_job = submit_and_verify(
    trainer_osft_after,
    label="C2 · MiniTrainer (osft) — With callbacks (MiniTrainerSmokeLogger)",
    expect_callbacks=True,
    result_key="osft_after",
    callback_markers=MINI_CALLBACK_MARKERS,
)

---
# Verification summary

In [ ]:
ROWS = [
    ("Unsloth", "lora_sft", "Baseline (no callbacks=)", "unsloth_before"),
    ("Unsloth", "lora_sft", "With callbacks (UnslothSmokeLogger)", "unsloth_after"),
    ("InstructLab", "sft", "Baseline (no callbacks=)", "sft_before"),
    ("InstructLab", "sft", "With callbacks (InstructLabSmokeLogger)", "sft_after"),
    ("MiniTrainer", "osft", "Baseline (no callbacks=)", "osft_before"),
    ("MiniTrainer", "osft", "With callbacks (MiniTrainerSmokeLogger)", "osft_after"),
]

print("\n" + "=" * 72)
print("Verification summary — Unified Training Hub Callbacks")
print("=" * 72)
print(f"{'Backend':<14} {'Algo':<10} {'Scenario':<35} {'Result':<8}")
print("-" * 72)

all_pass = True
for backend, algo, scenario, key in ROWS:
    if key not in VERIFICATION_RESULTS:
        status = "SKIP"
        all_pass = False
    else:
        status = "PASS" if VERIFICATION_RESULTS[key] else "FAIL"
        all_pass = all_pass and VERIFICATION_RESULTS[key]
    print(f"{backend:<14} {algo:<10} {scenario:<35} {status:<8}")

print("-" * 72)
if all_pass and len(VERIFICATION_RESULTS) == len(ROWS):
    print("\nAll six scenarios passed.")
    print("Callback markers verified per backend: UNSLOTH | ILAB | MINI.")
else:
    print("\nOne or more scenarios failed. Review the rows marked FAIL or SKIP above.")
    raise RuntimeError("Callback smoke verification failed — see summary table above.")

## Troubleshooting

| Symptom | Fix |
|---------|-----|
| `callbacks` missing on `TrainingHubTrainer` | Re-run §1 (SDK install) |
| `inspect.getsource` error | Callback must be in `hub_callback_smoke.py`, not inline |
| Baseline shows unexpected markers | Old job logs cached — check correct job name |
| OSFT `NameError: is_main_process` | Ensure `MINI_TRAINER_REF` in `packages_to_install` |
| SFT/OSFT MLflow error | `mlflow_tracking_uri=""` already set in func_args |
| 403 on runtimes | Grant notebook SA RBAC for TrainJobs |